<a href="https://colab.research.google.com/github/Metamask-ctrl/app/blob/main/ComfyUIonColab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#`mountUmount(`<font size="3px" color="#01c968">`Gdrive`</font>`)`



In [ ]:
#@markdown <br><center><img src='https://upload.wikimedia.org/wikipedia/commons/thumb/d/da/Google_Drive_logo.png/600px-Google_Drive_logo.png' height="50" alt="Gdrive-logo"/></center>
#@markdown <center><h3>Mount Gdrive to /content/drive</h3></center><br>
MODE = "MOUNT" #@param ["MOUNT", "UNMOUNT"]
#Mount your Gdrive!
from google.colab import drive
drive.mount._DEBUG = False
if MODE == "MOUNT":
  drive.mount('/content/drive', force_remount=True)
elif MODE == "UNMOUNT":
  try:
    drive.flush_and_unmount()
  except ValueError:
    pass
  get_ipython().system_raw("rm -rf /root/.config/Google/DriveFS")

#`Setup And Update ComfyUI & BAGEL Nodes`



In [ ]:
from pathlib import Path

OPTIONS = {}

DRIVE_PATH = ""  # @param {type:"string"}
UPDATE_COMFY_UI = True  #@param {type:"boolean"}
WORKSPACE = '/content/ComfyUI'
OPTIONS['UPDATE_COMFY_UI'] = UPDATE_COMFY_UI

if DRIVE_PATH:
    WORKSPACE = DRIVE_PATH+"/ComfyUI"
    %cd {DRIVE_PATH}

![ ! -d WORKSPACE ] && echo -= Initial setup ComfyUI =- && git clone https://github.com/comfyanonymous/ComfyUI
%cd $WORKSPACE

if OPTIONS['UPDATE_COMFY_UI']:
  !echo -= Updating ComfyUI =-
  !git pull

!echo -= Install dependencies =-
!pip install xformers!=0.0.18 -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cu121

# Install BAGEL & Custom Nodes
%cd {WORKSPACE}/custom_nodes
!git clone https://github.com/samm-ai/ComfyUI-Bagel.git
!git clone https://github.com/MienStudio/comfyui_mienodes.git

%cd {WORKSPACE}/custom_nodes/ComfyUI-Bagel
!pip install -r requirements.txt
!pip install bitsandbytes accelerate huggingface_hub

%cd {WORKSPACE}

# `BAGEL-7B Model Download`

## Download BAGEL-7B Model

In [ ]:
%cd {WORKSPACE}
# Install Aria2
!apt-get -y install -qq aria2

import os
from huggingface_hub import snapshot_download

# Create target folder for BAGEL models
model_dir = f"{WORKSPACE}/models/bagel/ByteDance-Seed/BAGEL-7B-MoT"
os.makedirs(model_dir, exist_ok=True)

print("Downloading BAGEL-7B-MoT Model from HuggingFace...")
snapshot_download(
    repo_id='ByteDance-Seed/BAGEL-7B-MoT',
    local_dir=model_dir,
    max_workers=16
)
print("✅ BAGEL-7B-MoT Download Complete!")

%cd {WORKSPACE}

## LIST BAGEL MODELS

In [ ]:
!ls -al ./models/bagel/ByteDance-Seed/BAGEL-7B-MoT/

# `START ComfyUI  & Expose Server (MANUAL)`

## Download Prerequisits

In [ ]:
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

In [ ]:
!nohup python main.py --listen --enable-cors-header --dont-print-server > comfyui.log 2>&1 &

# `START ComfyUI & Expose Server`

## Download Prerequisits

In [ ]:
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

## CF Tunnel

In [ ]:
import subprocess
import time
import socket
from google.colab import userdata

# 1. 从 Colab Secrets 中获取名为 CLOUDFLARE_TOKEN 的密钥
TUNNEL_TOKEN = userdata.get('CLOUDFLARE_TOKEN')

# 2. 在后台启动 ComfyUI 并加上 --listen 和 --enable-cors-header 避免 403 错误
print("Starting ComfyUI in background...")
comfy_process = subprocess.Popen(
    ["python", "main.py", "--listen", "--enable-cors-header", "--dont-print-server"],
    stdout=open("comfyui.log", "w"),
    stderr=subprocess.STDOUT
)

# 3. 轮询等待 ComfyUI 端口 (8188) 启动就绪
port = 8188
while True:
    time.sleep(1)
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    result = sock.connect_ex(('127.0.0.1', port))
    sock.close()
    if result == 0:
        break

print("\nComfyUI loaded successfully! Launching your Cloudflare Tunnel...\n")

# 4. 在前台运行你自己的 cloudflared 隧道
p = subprocess.Popen(
    ["cloudflared", "tunnel", "run", "--token", TUNNEL_TOKEN]
)

p.wait()

## localtunnel

In [ ]:
# localtunnel
!npm install -g localtunnel

import subprocess
import threading
import time
import socket
import urllib.request

def iframe_thread(port):
  while True:
      time.sleep(0.5)
      sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
      result = sock.connect_ex(('127.0.0.1', port))
      if result == 0:
        break
      sock.close()
  print("\nComfyUI finished loading, trying to launch localtunnel (if it gets stuck here localtunnel is having issues)\n")

  print("The password/enpoint ip for localtunnel is:", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip("\n"))
  p = subprocess.Popen(["lt", "--port", "{}".format(port)], stdout=subprocess.PIPE)
  for line in p.stdout:
    print(line.decode(), end='')


threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()
%cd /content/ComfyUI
!python main.py --dont-print-server